# NBA production run

This is the NBA notebook; do not substitute `kaggle_mlb_run.ipynb`. It is orchestration only. It runs the standalone `nba-backend` pipeline, which reads NBA.com's public APIs, writes the normalized cache outside the repository, and prepares only `nba-backend/data_delivery` for human-reviewed publication. It never commits or pushes.


In [ ]:
import os
import subprocess
from pathlib import Path

EXPECTED_REPO = "andrewkemmer/sports_prediction_model"
REPO = Path("/kaggle/working/sports_prediction_model")
DELIVERY = Path("nba-backend/data_delivery")

os.environ["NBA_PUSH"] = "0"
# Keep the derived cache outside the checkout. Kaggle's working directory is
# ephemeral, but this also makes the scope explicit.
os.environ.setdefault("NBA_CACHE_DIR", "/kaggle/working/nba-cache")

if REPO.exists():
    subprocess.run(["rm", "-rf", str(REPO)], check=True)
subprocess.run(["git", "clone", "--depth", "1",
                f"https://github.com/{EXPECTED_REPO}.git", str(REPO)], check=True)
assert (REPO / "nba-backend/backend/master_pipeline.py").is_file()
print(f"cloned {EXPECTED_REPO}")


In [ ]:
subprocess.run(["pip", "install", "-q", "-r", "nba-backend/backend/requirements-kaggle.txt"], cwd=REPO, check=True)
print("NBA dependencies installed")

In [ ]:
# The pipeline reads NBA.com's public APIs and its own cache. There is no
# dataset to resolve and nothing to mount: on a host that refuses NBA.com it
# falls back to ESPN's schedules and per-game box scores, which is slow but
# complete, and it refuses outright rather than publish without player detail.
cmd = ["python", "nba-backend/backend/master_pipeline.py"]
result = subprocess.run(cmd, cwd=REPO, env=os.environ.copy())
if result.returncode != 0:
    raise SystemExit(f"NBA pipeline failed with exit code {result.returncode}")
print("NBA production pipeline completed")


In [ ]:
# Delivery boundary audit: stage ONLY the NBA delivery directory. This is
# intentionally not a commit and not a push; publication remains a
# separate human-reviewed action.
staged = subprocess.run(
    ["git", "diff", "--name-only", "--", str(DELIVERY)],
    cwd=REPO, capture_output=True, text=True, check=True,
).stdout.splitlines()
outside = [p for p in staged if not p.replace('\\', '/').startswith('nba-backend/data_delivery/')]
if outside:
    raise RuntimeError(f"delivery scope violation: {outside}")
subprocess.run(["git", "add", "--", str(DELIVERY)], cwd=REPO, check=True)
cached = subprocess.run(
    ["git", "diff", "--cached", "--name-only"], cwd=REPO,
    capture_output=True, text=True, check=True,
).stdout.splitlines()
outside_cached = [p for p in cached if not p.replace('\\', '/').startswith('nba-backend/data_delivery/')]
if outside_cached:
    raise RuntimeError(f"staged scope violation: {outside_cached}")
print(f"staged {len(cached)} NBA delivery files; no commit or push performed")